# E1 · PCA redone on the test set: accuracy below chance

**Type**: Find the bug · **Level**: ●●○ independent · **Time**: about 40 minutes · **core path ★**

## Objective

Find out why a classifier trained on the components of the training set gets 0.13 accuracy (worse than chance, with 3 classes) on the test set: the colleague **redid the PCA on the test set**, so the components of the test set are different axes (and with different signs) from those of the training set.
Fix it with `transform` and with a `Pipeline`.

## Dataset

`load_wine`, stratified 70/30 split, standardized with the scaler learned on the training set: `Z_tr, Z_te`, `y_tr, y_te`. PCA with 5 components + `LogisticRegression`.

## Educational purpose

Recognize the most typical bug of "unsupervised" transformers: `fit_transform` on the test set. It produces no errors (the shapes are right) and the output numbers look reasonable, but they are coordinates in a **different frame of reference** from the one the model was trained on.

## Task

1. Run the colleague's cell: `bug_acc` is very low.
2. Compare the components: `cosines` = list of the dot products `train_pca.components_[i] @ test_pca.components_[i]` for the 5 components (where `test_pca` is the PCA redone on the test set) and `flipped_signs` = how many have a **negative** dot product. A value close to 1 means the same direction; negative, the opposite direction.
3. Fix it: apply to the test data the PCA **trained on the training set** (`transform`, not `fit_transform`). Save `ok_acc`.
4. Put everything in a `Pipeline` (`StandardScaler` → `PCA(5)` → `LogisticRegression`) on the raw data `X_tr_raw, X_te_raw` and save `pipeline_acc`. It must match `ok_acc`.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import load_wine
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = load_wine(return_X_y=True)
X_tr_raw, X_te_raw, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)
scaler = StandardScaler().fit(X_tr_raw)
Z_tr, Z_te = scaler.transform(X_tr_raw), scaler.transform(X_te_raw)

## Your work

### The colleague's code

In [ ]:
train_pca = PCA(5).fit(Z_tr)
P_tr = train_pca.transform(Z_tr)
test_pca = PCA(5).fit(Z_te)                     # "to get the components of the test set"
P_te = test_pca.transform(Z_te)
model = LogisticRegression(max_iter=1000).fit(P_tr, y_tr)
bug_acc = model.score(P_te, y_te)
print(f"test accuracy: {bug_acc:.3f}")

### Your diagnosis and the fix

In [ ]:
cosines = None
flipped_signs = None

In [ ]:
ok_acc = None
pipeline_acc = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`cosines = [float(train_pca.components_[i] @ test_pca.components_[i]) for i in range(5)]`. Fix: `model.score(train_pca.transform(Z_te), y_te)`. Pipeline: `make_pipeline(StandardScaler(), PCA(5), LogisticRegression(max_iter=1000)).fit(X_tr_raw, y_tr).score(X_te_raw, y_te)`.'):
    check(bug_acc < 0.5, "the colleague gets less than 0.5 with three classes", "don't modify the colleague's code")
    check(cosines is not None and len(cosines) == 5 and flipped_signs >= 2 and cosines[0] > 0.9 and min(cosines) < 0, "the first component is almost the same; at least two components have the opposite direction", "dot product between the rows of components_")
    check(ok_acc is not None and ok_acc > 0.9, "with transform (the same components as the training set) the accuracy goes back above 0.9", "train_pca.transform(Z_te)")
    check(pipeline_acc is not None and close(pipeline_acc, ok_acc), "the Pipeline gives exactly the same result", "make_pipeline(StandardScaler(), PCA(5), LogisticRegression())")
done()

---
The commented solution is in `solutions/E_find_the_bug/E1_pca_fitted_on_the_test_set.ipynb`. Open it after you pass the check, or after you have really tried.